# KD from BEATs into small students
Teacher: BEATs fold0 (frozen). Students: 16 small models in four groups, each trained one after another with logit distillation. Set `group_index` in Settings to pick the group for a molab session.
1. MobileViT-XXS/XS/S, EfficientViT-M0/M1. 2. EfficientViT-M2 to M5, DeiT-Tiny. 3. TinyViT-5M, PVTv2-B0, LeViT-128S, EdgeNeXt-S, SwiftFormer-XS. 4. CED-Tiny (reference).
Every softmax attention block gets the BEATs position bias: bucket + embedding table (one per head count, shared by the blocks), gated by q. EdgeNeXt (channel attention) and SwiftFormer (additive attention) have no token-to-token attention, so they get plain logit KD.
Recipe follows the best CED-Tiny run (val 0.629): no dropout, mixup 0.8 on the waveform (Beta 0.4), 64 mels, 500-step lr warmup, position tables initialised to zero, 30 epochs, patience 7.
Students in groups 1 to 3 are trained from scratch. CED-Tiny is the DeiT-Tiny architecture with the CED recipe, initialised from the CED-Tiny AudioSet checkpoint (`ced_pretrained`, set it to False to train from scratch). The frozen teacher is loaded from its checkpoint.
Logit KD is the baseline; the `+qkv` variant adds the BEATs q, k, v distillation (row-wise relation matrices of every block, teacher heads grouped onto the student heads, teacher frequency patches pooled onto the student grid) for the ViT students that have softmax attention blocks (DeiT-Tiny, CED-Tiny). Model selection (best epoch, early stop) uses the full validation set. The tables also report `validation_clean`: the validation samples whose noise ytid is in AudioSet eval and outside its train splits, which pretrained models cannot have seen (list in `validation_clean.csv`).

Current run: group 4 only (`group_index = 3`), CED-Tiny with BEATs logit KD plus q, k, v distillation (`kd_variants = ("logit+qkv",)`). Set `group_index` back to 0 to train the other groups.

## 1. Setup

In [ ]:
import json
import re
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import timm
import torch
import torch.nn.functional as F
import yaml
from timm.models import levit, pvt_v2, tiny_vit
from torch import Tensor, nn
from torch.utils.data import Subset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from scripts.load_dataset import build_dataloader, subsample
from scripts.run_reviewkd_deit import CONFIG, build_cached_loaders, load_teacher_checkpoint
from src.loss.beat_attention_kd import group_heads, relation_kl, relation_logits, relation_mse
from src.model.baseline.vit_bias import GATE_HIDDEN, MAX_DISTANCE, NUM_BUCKETS, BiasAttention, SharedBias, ViTSBias, load_ced_tiny
from src.model.beat.attention import relative_position_bucket
from src.model.kdmoe.frontend import LogMel, SpecAugment
from src.train import ReviewKDTrainConfig
from src.train.reviewkd_trainer import NINE_METRICS, snr_scores
from src.utils.seed import set_seed

In [ ]:
@dataclass(frozen=True)
class Settings:
    student_groups: tuple[tuple[str, ...], ...] = (
        ("mobilevit_xxs", "mobilevit_xs", "mobilevit_s", "efficientvit_m0", "efficientvit_m1"),
        ("efficientvit_m2", "efficientvit_m3", "efficientvit_m4", "efficientvit_m5", "deit_tiny_patch16_224"),
        ("tiny_vit_5m_224", "pvt_v2_b0", "levit_128s", "edgenext_small", "swiftformer_xs"),
        ("ced_tiny",),
    )
    group_index: int = 3  # one group per molab session; 3 = CED-Tiny only (KD from BEATs)
    teacher_checkpoint: Path = ROOT / "pretrained" / "beat-ft" / "beats_teacher_fold0_weights.pt"
    drive_noise_zip: str = "FA26_Capstone/Truong/beat-mv-kd-v3/nhat/nhat_cache.zip"
    noise_cache_dir: Path = ROOT / "cache" / "nhat"
    output_dir: Path = ROOT / "output" / "kd-smodels"

    # CED-Tiny recipe (kd-v4, val 0.629): no dropout, mixup, 64 mels, warmup, zero-initialised position tables
    num_epochs: int = 30
    early_stop_patience: int = 7
    learning_rate: float = 1e-4
    warmup_steps: int = 500
    lr_decay_factor: float = 0.5
    lr_decay_patience: int = 1
    min_learning_rate: float = 1e-6
    weight_decay: float = 0.1
    dropout: float = 0.0  # CED-Tiny collapsed to chance with 0.2
    drop_path: float = 0.0
    use_post_ln: bool = False  # BEATs-style post-LN alpha, DeiT-Tiny only
    mixup_probability: float = 0.8
    mixup_beta: float = 0.4
    student_mels: int = 64
    pretrained: bool = False  # groups 1 to 3 are trained from scratch
    ced_pretrained: bool = True  # CED-Tiny starts from its AudioSet weights (mispeech/ced-tiny); False trains it from scratch

    batch_size: int = 32
    num_workers: int = 4
    max_samples: int | None = None  # set small for a smoke test
    train_eval_clips: int = 2000
    clean_validation_file: str = "validation_clean.csv"  # validation samples whose noise ytid is outside the AudioSet train splits; report only

    num_classes: int = 36
    temperature: float = 4.0
    kd_weight: float = 1.0
    qkv_weight: float = 0.1  # weight of the q, k, v relation loss in the "+qkv" runs (1.0 dominated the logit loss; starts near 32 from the AudioSet weights)
    qkv_distance: str = "mse"  # "mse" or "kl" between student and teacher relation matrices
    kd_variants: tuple[str, ...] = ("logit+qkv",)  # ("logit", "logit+qkv") also runs the baseline; "+qkv" runs only on students with softmax-attention ViT blocks
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    square_input_size: int = 224  # mel is resized to this for every backbone with a fixed-size input
    mel_native_prefixes: tuple[str, ...] = ("mobilevit", "deit", "ced")
    seed: int = 42

    @property
    def student_names(self) -> tuple[str, ...]:
        return self.student_groups[self.group_index]


settings = Settings()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(settings.seed)
print(f"device: {device}")

## 2. Inputs
Fetched once, skipped when present: `mixture-36l` from Kaggle (`KAGGLE_ACCESS_TOKEN`) and the noise cache from Google Drive (`fs`; on molab it comes from the login cell).

In [ ]:
import os
import shutil

from scripts import download_dataset as dl
from scripts.load_dataset import MIXTURE_KEY

COPY_BUFFER_BYTES = 64 * 1024 * 1024
NOISE_SPLITS = ("train", "validation")
fs = None  # Google Drive filesystem; set it before running when the cache is not on disk


def noise_cache_ready(cache_dir: Path) -> bool:
    return all((cache_dir / f"{split}.done").is_file() for split in NOISE_SPLITS)


def fetch_noise_cache(cache_dir: Path) -> Path:
    if not noise_cache_ready(cache_dir):
        archive_path = ROOT / "nhat_cache.zip"
        with fs.open(settings.drive_noise_zip, "rb") as source, open(archive_path, "wb") as target:
            shutil.copyfileobj(source, target, COPY_BUFFER_BYTES)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(ROOT)
        archive_path.unlink()
    if not noise_cache_ready(cache_dir):
        raise FileNotFoundError(f"{cache_dir} is incomplete")
    return cache_dir


DATASET_ROOT = dl.ensure_dataset(MIXTURE_KEY, ROOT / "datasets")
NOISE_CACHE_DIR = fetch_noise_cache(settings.noise_cache_dir)
print(f"dataset: {DATASET_ROOT}\nnoise cache: {NOISE_CACHE_DIR}")

## 3. Data

In [ ]:
raw_config = yaml.safe_load(open(CONFIG, encoding="utf-8"))
train_config = ReviewKDTrainConfig.from_dict({**raw_config["train"], "batch_size": settings.batch_size})
loaders = build_cached_loaders(raw_config, train_config, NOISE_CACHE_DIR, settings.max_samples, settings.num_workers)
train_eval_loader = build_dataloader(
    subsample(loaders["train"].dataset, settings.train_eval_clips), train_config.eval_batch_size, False, settings.num_workers,
)
clean_ids = set(pd.read_csv(DATASET_ROOT / settings.clean_validation_file)["sample_id"])
clean_indices = [index for index, row in enumerate(loaders["validation"].dataset.base.rows) if row["sample_id"] in clean_ids]
clean_validation_loader = build_dataloader(
    Subset(loaders["validation"].dataset, clean_indices), train_config.eval_batch_size, False, settings.num_workers,
)
print({split: len(loaders[split].dataset) for split in ("train", "validation")}, "| clean validation:", len(clean_indices))

## 4. Teacher

In [ ]:
front, teacher, teacher_meta = load_teacher_checkpoint(settings.teacher_checkpoint, device, use_frcrn=False)
teacher.freeze()
teacher_params = sum(p.numel() for p in teacher.parameters())
print(f"teacher: BEATs fold0 | {teacher_params / 1e6:.1f}M params | epoch {teacher_meta['epoch']}")

## 5. Students

In [ ]:
class BiasedCascadedAttention(nn.Module):
    """EfficientViT cascaded group attention with the BEATs gated bucket bias instead of its own offset bias."""

    def __init__(self, base: nn.Module, table: nn.Embedding) -> None:
        super().__init__()
        self.num_heads, self.key_dim, self.val_dim, self.scale = base.num_heads, base.key_dim, base.val_dim, base.scale
        self.qkvs, self.dws, self.proj = base.qkvs, base.dws, base.proj
        self.relative_attention_bias = table
        self.grep_linear = nn.Linear(self.key_dim, GATE_HIDDEN)
        self.grep_a = nn.Parameter(torch.ones(self.num_heads))

    def position_bias(self, tokens: int, device) -> Tensor:
        pos = torch.arange(tokens, device=device)
        bucket = relative_position_bucket(pos[None, :] - pos[:, None], NUM_BUCKETS, MAX_DISTANCE)
        return self.relative_attention_bias(bucket).permute(2, 0, 1)  # (heads, N, N)

    def forward(self, x: Tensor) -> Tensor:
        B, C, H, W = x.shape
        position_bias = self.position_bias(H * W, x.device)
        feats_in = x.chunk(self.num_heads, dim=1)
        feats_out, feat = [], feats_in[0]
        for head, (qkv, dw) in enumerate(zip(self.qkvs, self.dws)):
            if head > 0:
                feat = feat + feats_in[head]
            feat = qkv(feat)
            q, k, v = feat.view(B, -1, H, W).split([self.key_dim, self.key_dim, self.val_dim], dim=1)
            q, k, v = dw(q).flatten(2), k.flatten(2), v.flatten(2)
            gate = torch.sigmoid(self.grep_linear(q.transpose(1, 2))).view(B, H * W, 2, GATE_HIDDEN // 2).sum(-1)
            gate_a, gate_b = gate.chunk(2, dim=-1)  # (B, N, 1)
            scale = gate_a * (gate_b * self.grep_a[head] - 1.0) + 2.0
            attn = (q * self.scale).transpose(-2, -1) @ k + scale * position_bias[head]
            feat = (v @ attn.softmax(dim=-1).transpose(-2, -1)).view(B, self.val_dim, H, W)
            feats_out.append(feat)
        return self.proj(torch.cat(feats_out, 1))


class BiasedQueryKeyAttention(nn.Module):
    """Base for softmax attentions that take q, k, v from a timm layer and add the BEATs gated bias (flattened-grid positions)."""

    def __init__(self, base: nn.Module, table: nn.Embedding, num_heads: int, query_dim: int) -> None:
        super().__init__()
        self.base = base
        self.relative_attention_bias = table
        self.num_heads = num_heads
        self.grep_linear = nn.Linear(query_dim, GATE_HIDDEN)
        self.grep_a = nn.Parameter(torch.ones(1, num_heads, 1, 1))
        if hasattr(base, "attention_biases"):
            base.attention_biases = None  # replaced by the BEATs table

    def position_bias(self, q: Tensor, query_index: Tensor, key_index: Tensor) -> Tensor:
        B, H, N, _ = q.shape
        bucket = relative_position_bucket(key_index[None, :] - query_index[:, None], NUM_BUCKETS, MAX_DISTANCE)
        bias = self.relative_attention_bias(bucket).permute(2, 0, 1).unsqueeze(0)  # (1, H, Nq, Nk)
        gate = torch.sigmoid(self.grep_linear(q).view(B, H, N, 2, GATE_HIDDEN // 2).sum(-1))
        gate_a, gate_b = gate.chunk(2, dim=-1)
        return (gate_a * (gate_b * self.grep_a - 1.0) + 2.0) * bias


def grid_index(rows: int, cols: int, stride: int, width: int, device) -> Tensor:
    rows_index = torch.arange(0, rows, stride, device=device)
    cols_index = torch.arange(0, cols, stride, device=device)
    return (rows_index[:, None] * width + cols_index[None, :]).flatten()


class BiasedTinyViTAttention(BiasedQueryKeyAttention):
    def __init__(self, base: nn.Module, table: nn.Embedding) -> None:
        super().__init__(base, table, base.num_heads, base.key_dim)

    def forward(self, x: Tensor) -> Tensor:
        base = self.base
        B, N, _ = x.shape
        q, k, v = base.qkv(base.norm(x)).view(B, N, base.num_heads, -1).split([base.key_dim, base.key_dim, base.val_dim], dim=3)
        q, k, v = q.permute(0, 2, 1, 3), k.permute(0, 2, 1, 3), v.permute(0, 2, 1, 3)
        index = torch.arange(N, device=x.device)
        bias = self.position_bias(q, index, index).to(q.dtype)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias)
        return base.proj(out.transpose(1, 2).reshape(B, N, base.out_dim))


class BiasedPVTAttention(BiasedQueryKeyAttention):
    def __init__(self, base: nn.Module, table: nn.Embedding) -> None:
        super().__init__(base, table, base.num_heads, base.head_dim)

    def forward(self, x: Tensor, feat_size: list[int]) -> Tensor:
        base = self.base
        B, N, C = x.shape
        height, width = feat_size
        q = base.q(x).reshape(B, N, base.num_heads, -1).permute(0, 2, 1, 3)
        if base.sr is None:
            reduced, key_index = x, torch.arange(N, device=x.device)
        else:
            ratio = base.sr.kernel_size[0]
            reduced = base.norm(base.sr(x.permute(0, 2, 1).reshape(B, C, height, width)).reshape(B, C, -1).permute(0, 2, 1))
            key_index = grid_index(height // ratio * ratio, width // ratio * ratio, ratio, width, x.device)
        k, v = base.kv(reduced).reshape(B, -1, 2, base.num_heads, base.head_dim).permute(2, 0, 3, 1, 4).unbind(0)
        bias = self.position_bias(q, torch.arange(N, device=x.device), key_index).to(q.dtype)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias)
        return base.proj_drop(base.proj(out.transpose(1, 2).reshape(B, N, C)))


class BiasedLeViTAttention(BiasedQueryKeyAttention):
    def __init__(self, base: nn.Module, table: nn.Embedding) -> None:
        super().__init__(base, table, base.num_heads, base.key_dim)

    def forward(self, x: Tensor) -> Tensor:
        base = self.base
        B, N, _ = x.shape
        q, k, v = base.qkv(x).view(B, N, base.num_heads, -1).split([base.key_dim, base.key_dim, base.val_dim], dim=3)
        q, k, v = q.permute(0, 2, 1, 3), k.permute(0, 2, 1, 3), v.permute(0, 2, 1, 3)
        index = torch.arange(N, device=x.device)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=self.position_bias(q, index, index).to(q.dtype))
        return base.proj(out.transpose(1, 2).reshape(B, N, base.val_attn_dim))


class BiasedLeViTDownsample(BiasedQueryKeyAttention):
    def __init__(self, base: nn.Module, table: nn.Embedding) -> None:
        super().__init__(base, table, base.num_heads, base.key_dim)

    def forward(self, x: Tensor) -> Tensor:
        base = self.base
        B, N, _ = x.shape
        rows, cols = base.resolution
        k, v = base.kv(x).view(B, N, base.num_heads, -1).split([base.key_dim, base.val_dim], dim=3)
        k, v = k.permute(0, 2, 1, 3), v.permute(0, 2, 1, 3)
        q = base.q(x).view(B, -1, base.num_heads, base.key_dim).permute(0, 2, 1, 3)
        query_index = grid_index(rows, cols, base.stride, cols, x.device)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=self.position_bias(q, query_index, torch.arange(N, device=x.device)).to(q.dtype))
        return base.proj(out.transpose(1, 2).reshape(B, -1, base.val_attn_dim))


ATTENTION_WRAPPERS = (
    (tiny_vit.Attention, BiasedTinyViTAttention),
    (pvt_v2.Attention, BiasedPVTAttention),
    (levit.Attention, BiasedLeViTAttention),
    (levit.AttentionDownsample, BiasedLeViTDownsample),
)


def attach_beats_bias(backbone: nn.Module) -> None:
    """Replace every softmax position attention of a timm backbone with its BEATs-biased version.

    EdgeNeXt (channel attention) and SwiftFormer (additive attention) have no token-to-token attention to bias.
    """
    for module in list(backbone.modules()):
        if hasattr(module, "transformer"):  # MobileViT block: one table per block, owned by its first layer
            shared = SharedBias()
            for index, layer in enumerate(module.transformer):
                layer.attn = BiasAttention(layer.attn, shared, owns_table=index == 0, has_cls=False)
    tables: dict[int, nn.Embedding] = {}  # one table per head count, shared by all blocks of a backbone
    for module in list(backbone.modules()):
        if hasattr(module, "attn") and type(module.attn).__name__ == "CascadedGroupAttention":
            heads = module.attn.num_heads
            tables.setdefault(heads, nn.Embedding(NUM_BUCKETS, heads))
            module.attn = BiasedCascadedAttention(module.attn, tables[heads])
        for child_name, child in list(module.named_children()):
            for source, wrapper in ATTENTION_WRAPPERS:
                if isinstance(child, source):
                    tables.setdefault(child.num_heads, nn.Embedding(NUM_BUCKETS, child.num_heads))
                    setattr(module, child_name, wrapper(child, tables[child.num_heads]))


class SpectrogramStudent(nn.Module):
    """Noise waveform -> log-mel -> SpecAugment (train only) -> backbone."""

    def __init__(self, backbone: nn.Module, input_size: int | None = None) -> None:
        super().__init__()
        model_cfg = raw_config["model"]
        self.frontend = LogMel(**{**raw_config["frontend"], "n_mels": settings.student_mels})
        self.spec_augment = SpecAugment(
            freq_masks=model_cfg["specaug_freq_masks"],
            freq_width=max(1, model_cfg["specaug_freq_width"] * settings.student_mels // raw_config["frontend"]["n_mels"]),
            time_masks=model_cfg["specaug_time_masks"], time_width=model_cfg["specaug_time_width"],
        )
        self.backbone = backbone
        self.input_size = input_size

    def forward(self, wave: Tensor) -> Tensor:
        with torch.autocast(wave.device.type, enabled=False):
            mel, _ = self.frontend(wave.float())
        mel = self.spec_augment(mel)
        if self.input_size is not None:
            mel = F.interpolate(mel, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        return self.backbone(mel)


def zero_position_tables(backbone: nn.Module) -> None:
    for module in backbone.modules():
        if isinstance(getattr(module, "relative_attention_bias", None), nn.Embedding):
            module.relative_attention_bias.weight.data.zero_()


CED_TINY = "ced_tiny"
CED_ARCH = "deit_tiny_patch16_224"


def build_student(name: str, settings: Settings) -> SpectrogramStudent:
    if name.startswith("deit") or name == CED_TINY:
        is_ced = name == CED_TINY
        backbone = ViTSBias(
            num_classes=settings.num_classes, freq_bins=settings.student_mels, dropout=settings.dropout,
            attn_dropout=settings.dropout, drop_path=settings.drop_path, post_ln=settings.use_post_ln,
            arch=CED_ARCH if is_ced else name, pretrained=settings.pretrained and not is_ced,
        )
        if is_ced and settings.ced_pretrained:
            load_ced_tiny(backbone)
        zero_position_tables(backbone)
        return SpectrogramStudent(backbone)
    backbone = timm.create_model(
        name, pretrained=settings.pretrained, in_chans=1, num_classes=settings.num_classes, drop_rate=settings.dropout,
    )
    attach_beats_bias(backbone)
    zero_position_tables(backbone)
    input_size = None if name.startswith(settings.mel_native_prefixes) else settings.square_input_size
    return SpectrogramStudent(backbone, input_size)


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6

In [ ]:
params_preview = pd.Series({name: count_params_m(build_student(name, settings)) for name in settings.student_names}, name="params (M)")
print(params_preview.round(2).to_string())

## 6. Training and evaluation

In [ ]:
@torch.no_grad()
def evaluate(student: SpectrogramStudent, loader) -> dict[str, float]:
    student.eval()
    probabilities, labels, snrs = [], [], []
    for batch in tqdm(loader, desc="eval", leave=False):
        with torch.autocast(device.type, dtype=torch.bfloat16):
            logits = student(front(batch["waveform"].to(device)))
        probabilities.append(logits.float().softmax(-1).cpu())
        labels.append(batch["target"].argmax(dim=1))
        snrs.append(batch["snr_db"])
    return snr_scores(torch.cat(probabilities).numpy(), torch.cat(labels).numpy(), torch.cat(snrs).numpy(), settings.num_classes)


def mixup(wave: Tensor, target: Tensor) -> tuple[Tensor, Tensor]:
    if torch.rand(()) >= settings.mixup_probability:
        return wave, target
    beta = torch.distributions.Beta(settings.mixup_beta, settings.mixup_beta)
    weight = beta.sample((wave.shape[0], 1)).to(wave.device)
    weight = torch.maximum(weight, 1 - weight)
    order = torch.randperm(wave.shape[0], device=wave.device)
    return weight * wave + (1 - weight) * wave[order], weight * target + (1 - weight) * target[order]


def distillation_loss(logits: Tensor, teacher_logits: Tensor, target: Tensor) -> Tensor:
    smoothed = target * (1 - settings.label_smoothing) + settings.label_smoothing / settings.num_classes
    cross_entropy = -(smoothed * F.log_softmax(logits, -1)).sum(-1).mean()
    t = settings.temperature
    kl = F.kl_div(F.log_softmax(logits / t, -1), F.softmax(teacher_logits / t, -1), reduction="batchmean")
    return cross_entropy + settings.kd_weight * t**2 * kl


RELATION_DISTANCE = {"mse": relation_mse, "kl": relation_kl}
teacher_attentions = [layer.self_attn for layer in teacher.backbone.encoder.layers]


def pool_frequency(x: Tensor, time_patches: int, freq_patches: int) -> Tensor:
    # teacher tokens are ordered (time, frequency): average neighbouring frequency patches onto the student grid
    batch, heads, tokens, dim = x.shape
    x = x.reshape(batch, heads, time_patches, freq_patches, tokens // (time_patches * freq_patches), dim)
    return x.mean(dim=4).reshape(batch, heads, time_patches * freq_patches, dim)


def record_attention(student: SpectrogramStudent, enabled: bool) -> None:
    student.backbone.set_record(enabled)
    for layer in teacher_attentions:
        layer.record = enabled
        if not enabled:
            layer.cache = {}


def qkv_loss(student: SpectrogramStudent) -> Tensor:
    # row-wise relation matrices of q, k and v: student heads against grouped teacher heads, averaged over blocks
    vit = student.backbone
    distance = RELATION_DISTANCE[settings.qkv_distance]
    total = 0.0
    for own, reference in zip(vit.attentions, teacher_attentions, strict=True):
        heads = own.cache["q"].shape[1]
        for key in "qkv":
            with torch.no_grad():
                target = group_heads(relation_logits(pool_frequency(reference.cache[key], vit.time_patches, vit.freq_patches)), heads)
            total = total + distance(relation_logits(own.cache[key][:, :, 1:]), target) / 3  # token 0 is CLS
    return total / len(teacher_attentions)


def train_epoch(student: SpectrogramStudent, optimizer, epoch: int, name: str, base_lr: float, first_step: int, qkv_weight: float = 0.0) -> tuple[float, int, float, float]:
    student.train()
    record_attention(student, qkv_weight > 0)
    total_loss, total_qkv, total_norm, seen, step = 0.0, 0.0, 0.0, 0, first_step
    progress = tqdm(loaders["train"], desc=f"{name} epoch {epoch}/{settings.num_epochs}", leave=False)
    for batch in progress:
        wave, target = mixup(front(batch["waveform"].to(device)), batch["target"].to(device).float())
        labels = target
        for group in optimizer.param_groups:
            group["lr"] = base_lr * min(1.0, (step + 1) / settings.warmup_steps)
        with torch.autocast(device.type, dtype=torch.bfloat16):
            logits = student(wave).float()
            with torch.no_grad():
                teacher_logits = teacher(wave).logits.float()
            loss = distillation_loss(logits, teacher_logits, labels)
        if qkv_weight > 0:
            relation = qkv_loss(student)
            total_qkv += relation.item() * len(labels)
            loss = loss + qkv_weight * min(1.0, (step + 1) / settings.warmup_steps) * relation  # ramped in with the lr warmup
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        total_norm += nn.utils.clip_grad_norm_(student.parameters(), settings.grad_clip).item() * len(labels)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        seen += len(labels)
        step += 1
        progress.set_postfix(loss=f"{total_loss / seen:.4f}")
    record_attention(student, False)
    return total_loss / seen, step, total_qkv / seen, total_norm / seen


def train_student(name: str, qkv_weight: float = 0.0, run_name: str | None = None) -> SpectrogramStudent:
    run_name = run_name or name
    student = build_student(name, settings).to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=settings.learning_rate, weight_decay=settings.weight_decay)
    run_dir = settings.output_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_val, stale, history = -1.0, 0, []
    base_lr, lr_stale, lr_best, step = settings.learning_rate, 0, -1.0, 0
    for epoch in range(1, settings.num_epochs + 1):
        train_loss, step, qkv_term, grad_norm = train_epoch(student, optimizer, epoch, run_name, base_lr, step, qkv_weight)
        val_acc = evaluate(student, loaders["validation"])["macro_acc"]
        lr = optimizer.param_groups[0]["lr"]
        if val_acc > lr_best:
            lr_best, lr_stale = val_acc, 0
        else:
            lr_stale += 1
            if lr_stale >= settings.lr_decay_patience:
                base_lr, lr_stale = max(base_lr * settings.lr_decay_factor, settings.min_learning_rate), 0
        history.append({"epoch": epoch, "train_loss": train_loss, "qkv": qkv_term, "grad_norm": grad_norm, "val_macro_acc": val_acc, "lr": lr})
        (run_dir / "history.json").write_text(json.dumps(history, indent=2))
        print(f"{run_name} epoch {epoch:2d} | loss {train_loss:.4f} | qkv {qkv_term:.4f} | grad norm {grad_norm:.2f} | val macro_acc {val_acc:.4f} | lr {lr:.2e}")
        if val_acc > best_val:
            best_val, stale = val_acc, 0
            torch.save(student.state_dict(), run_dir / "best.pt")
            continue
        stale += 1
        if stale >= settings.early_stop_patience:
            print(f"{run_name}: early stop after epoch {epoch}")
            break
    student.load_state_dict(torch.load(run_dir / "best.pt", map_location=device))
    return student

## 7. Results table

In [ ]:
def snr_levels(scores: dict[str, float]) -> list[int]:
    pattern = re.compile(rf"^{NINE_METRICS[0]}_snr_(-?\d+)$")
    return sorted(int(match.group(1)) for key in scores if (match := pattern.match(key)))


def score_rows(name: str, scores: dict[str, float]) -> pd.DataFrame:
    rows = {"all": {metric: scores[metric] for metric in NINE_METRICS}}
    for snr in snr_levels(scores):
        rows[f"{snr} dB"] = {metric: scores[f"{metric}_snr_{snr}"] for metric in NINE_METRICS}
    table = pd.DataFrame.from_dict(rows, orient="index", columns=list(NINE_METRICS))
    table.index = pd.MultiIndex.from_product([[name], table.index], names=["model", "snr"])
    return table


eval_loaders = {"train": train_eval_loader, "validation": loaders["validation"], "validation_clean": clean_validation_loader}
results = {split: pd.DataFrame() for split in eval_loaders}


def append_results(name: str, student: SpectrogramStudent) -> None:
    for split, loader in eval_loaders.items():
        results[split] = pd.concat([results[split], score_rows(name, evaluate(student, loader))])
        results[split].to_csv(settings.output_dir / f"results_{split}_group{settings.group_index + 1}.csv")


def show_results() -> None:
    for split, table in results.items():
        print(f"\n=== {split} ({len(table)} rows) ===")
        print(table.round(4).to_string())

## 8. Run all students

In [ ]:
settings.output_dir.mkdir(parents=True, exist_ok=True)

for name in settings.student_names:
    for variant in settings.kd_variants:
        use_qkv = variant.endswith("+qkv")
        run_name = f"{name}+qkv" if use_qkv else name
        print(f"\n{'=' * 20} {run_name} {'=' * 20}")
        student = build_student(name, settings)
        if use_qkv and not hasattr(student.backbone, "attentions"):
            print(f"{name}: no softmax-attention ViT blocks, skipping {variant}")
            continue
        print(student.backbone)
        print(f"params: {count_params_m(student):.2f}M")
        trained = train_student(name, settings.qkv_weight if use_qkv else 0.0, run_name)
        append_results(run_name, trained)
        show_results()

## 9. Merge groups

In [ ]:
for split in results:
    parts = [pd.read_csv(path, index_col=[0, 1]) for path in sorted(settings.output_dir.glob(f"results_{split}_group*.csv"))]
    merged = pd.concat(parts)
    merged.to_csv(settings.output_dir / f"results_{split}_all.csv")
    print(f"\n=== {split} ({len(merged)} rows) ===")
    print(merged.round(4).to_string())